# DRVI Ig-Gene / Timepoint Analysis

Examines immunoglobulin genes (IGH*/IGL*) across the treatment timepoints (TP1-TP4 or control): heatmap of mean expression per timepoint, UMAP grid of selected genes across timepoints, and violin plots (cell type x timepoint).

Uses the original (uncurated) dataset `data_for_practicum_post_integration.h5ad` — independent of the curation in `2_D_drvi_curation_ig_genes.ipynb`.

## 1. IGH*/IGL* genes across timepoints (heatmap)

*Source: `2_9_a_drvi_ig_genes_by_timepoint.py`*

Compare IGH*/IGL* genes across timepoints.

`sample_id` encodes patient.timepoint (e.g. "m6.4" -> patient m6, TP4;
controls like "k9" have no timepoint, just a single draw -> "control").
For each IGH*/IGL* gene, the mean log1p-normalized expression per timepoint
is computed and shown as a gene x timepoint heatmap.

Optionally restrict to a cell type with --celltype-filter (e.g. 'B-cell' or
'Plasma Blast'), since Ig genes are barely expressed outside of B cells/
plasma cells and the signal would otherwise be diluted.

Usage:
    conda run -n mapra_cytokines python 2_9_a_drvi_ig_genes_by_timepoint.py \
        --gene-prefixes IGH,IGL --celltype-filter B-cell

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
import argparse
args = argparse.Namespace(
    drvi_input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    gene_prefixes="IGH,IGL",
    celltype_key="cell_type_Scanorama",
    celltype_filter="",
    out_name="",
)

In [ ]:
print("=== Load data ===")
adata = sc.read_h5ad(args.drvi_input)
adata.X = adata.layers["log1p_norm"]

if args.celltype_filter:
    keep = (adata.obs[args.celltype_key] == args.celltype_filter).values
    print(f"Filter '{args.celltype_key}' == '{args.celltype_filter}': {keep.sum()} / {adata.n_obs} cells")
    adata = adata[keep].copy()
else:
    print(f"No cell-type filter — all {adata.n_obs} cells.")

### Derive timepoint from sample_id

In [ ]:
# "m6.4" -> "TP4" ; "k9" (no dot) -> "control"

sample_id = adata.obs["sample_id"].astype(str)
split = sample_id.str.split(".", n=1, expand=True)
timepoint = np.where(split[1].notna(), "TP" + split[1], "control")
adata.obs["timepoint"] = pd.Categorical(
    timepoint, categories=["control", "TP1", "TP2", "TP3", "TP4"], ordered=True
)
print("\nCells per timepoint:")
print(adata.obs["timepoint"].value_counts().sort_index())

### Collect IGH*/IGL* genes

In [ ]:
prefixes = [p.strip() for p in args.gene_prefixes.split(",") if p.strip()]
genes = sorted(g for g in adata.var_names if any(g.startswith(p) for p in prefixes))
print(f"\n{len(genes)} genes found for prefixes {prefixes}: {genes}")

### Mean expression per gene x timepoint

In [ ]:
expr = adata[:, genes].X
expr = expr.toarray() if hasattr(expr, "toarray") else np.asarray(expr)
expr_df = pd.DataFrame(expr, columns=genes, index=adata.obs_names)
expr_df["timepoint"] = adata.obs["timepoint"].values

mean_by_tp = expr_df.groupby("timepoint", observed=True)[genes].mean().T  # genes x timepoints
n_cells_by_tp = adata.obs["timepoint"].value_counts()

ct_suffix = f"_{args.celltype_filter.replace(' ', '_').replace('-', '')}" if args.celltype_filter else ""
out_csv = os.path.join(args.output_dir, f"ig_genes_by_timepoint{ct_suffix}.csv")
mean_by_tp.to_csv(out_csv)
print(f"\nSaved: {os.path.basename(out_csv)}")

### Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(6, max(6, 0.22 * len(genes))))
im = ax.imshow(mean_by_tp.values, aspect="auto", cmap="YlOrBr")
ax.set_xticks(range(mean_by_tp.shape[1]))
ax.set_xticklabels([f"{c}\n(n={n_cells_by_tp.get(c, 0)})" for c in mean_by_tp.columns], fontsize=8)
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes, fontsize=6)
ax.set_xlabel("Timepoint")
ax.set_ylabel("Gene")
title = f"IGH*/IGL* mean expression by timepoint"
if args.celltype_filter:
    title += f"  [{args.celltype_filter}]"
ax.set_title(title)
cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label("Mean log1p-normalized expression")

plt.tight_layout()
out_name = args.out_name or f"ig_genes_by_timepoint{ct_suffix}.png"
out_fig = os.path.join(args.output_dir, out_name)
plt.savefig(out_fig, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"Saved: {out_fig}")

## 2. IGH*/IGL* genes across timepoints (UMAP grid)

*Source: `2_9_b_drvi_ig_genes_umap_by_timepoint.py`*

UMAP grid: selected IGH*/IGL* genes x timepoints.

For each gene (row) and each timepoint (column), expression is shown on the
DRVI UMAP: background = all cells (gray), foreground = only cells from that
timepoint, colored by expression. The color scale is consistent per gene
across all timepoints (same vmin/vmax per row), so the panels within a row
are directly comparable.

Timepoint derived from sample_id (see 2_9_a): "m6.4" -> TP4,
"k9" (no dot) -> control.

Usage:
    conda run -n mapra_cytokines python 2_9_b_drvi_ig_genes_umap_by_timepoint.py \
        --genes IGLC2,IGLC3,IGHM,IGHG1,IGHG2,IGHG3,IGHD,IGHA1

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
import argparse
args = argparse.Namespace(
    drvi_input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    genes="IGLC2,IGLC3,IGHM,IGHG1,IGHG2,IGHG3,IGHD,IGHA1",
    timepoints="TP1,TP2,TP3,TP4",
    color_map="YlOrBr",
    vmax_percentile=99.5,
    out_name="ig_genes_umap_by_timepoint.png",
)

In [ ]:
genes = [g.strip() for g in args.genes.split(",") if g.strip()]

print("=== Load data ===")
adata = sc.read_h5ad(args.drvi_input)
adata.X = adata.layers["log1p_norm"]

embed_path = os.path.join(args.output_dir, "embed.h5ad")
embed = sc.read_h5ad(embed_path)
umap_df = pd.DataFrame(embed.obsm["X_umap"], index=embed.obs_names)
umap = umap_df.loc[adata.obs_names].values

missing = [g for g in genes if g not in adata.var_names]
if missing:
    print(f"WARNING: not found in the dataset, skipped: {missing}")
genes = [g for g in genes if g in adata.var_names]
if not genes:
    raise RuntimeError("None of the specified genes were found in the dataset.")

### Derive timepoint from sample_id

In [ ]:
sample_id = adata.obs["sample_id"].astype(str)
split = sample_id.str.split(".", n=1, expand=True)
timepoint = np.where(split[1].notna(), "TP" + split[1], "control")
timepoints = [t.strip() for t in args.timepoints.split(",") if t.strip()]
timepoint = pd.Categorical(timepoint, categories=["control", "TP1", "TP2", "TP3", "TP4"], ordered=True)

n_cells_by_tp = pd.Series(timepoint).value_counts()
print("Cells per timepoint:")
print(n_cells_by_tp.reindex(timepoints))

### Plot grid

In [ ]:
n_genes = len(genes)
n_tp = len(timepoints)
fig, axes = plt.subplots(n_genes, n_tp, figsize=(3.2 * n_tp, 3.0 * n_genes))
if n_genes == 1:
    axes = axes[np.newaxis, :]

for i, gene in enumerate(genes):
    expr = adata[:, gene].X
    expr = np.asarray(expr.todense()).flatten() if hasattr(expr, "todense") else np.asarray(expr).flatten()
    vmax = max(np.percentile(expr, args.vmax_percentile), 1e-6)

    for j, tp in enumerate(timepoints):
        ax = axes[i, j]
        mask = (timepoint == tp)

        ax.scatter(umap[:, 0], umap[:, 1], s=1, c="lightgray", alpha=0.4, linewidths=0, rasterized=True)
        sc_plot = ax.scatter(umap[mask, 0], umap[mask, 1], s=2, c=expr[mask],
                              cmap=args.color_map, vmin=0, vmax=vmax, linewidths=0, rasterized=True)
        if i == 0:
            ax.set_title(f"{tp}\n(n={int(mask.sum())})", fontsize=9)
        if j == 0:
            ax.set_ylabel(gene, fontsize=10, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])

    fig.colorbar(sc_plot, ax=axes[i, -1], fraction=0.046, pad=0.04)

plt.tight_layout()
out = os.path.join(args.output_dir, args.out_name)
plt.savefig(out, bbox_inches="tight", dpi=130)
plt.close("all")
print(f"\nSaved: {out}")

## 3. IGH*/IGL* genes: violin plot (cell type x timepoint)

*Source: `2_9_c_drvi_ig_genes_violin.py`*

Violin plot: IGH*/IGL* genes (facets) x cell type (x-axis), colored by
timepoint (hue).

Timepoint derived from sample_id as in 2_9_a/2_9_b ("m6.4" -> TP4,
"k9" -> control).

Usage:
    conda run -n mapra_cytokines python 2_9_c_drvi_ig_genes_violin.py \
        --genes IGLC2,IGLC3,IGHM,IGHG1,IGHG2,IGHG3,IGHD,IGHA1 \
        --timepoints TP1,TP2,TP3,TP4

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import argparse
args = argparse.Namespace(
    drvi_input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    genes="IGLC2,IGLC3,IGHM,IGHG1,IGHG2,IGHG3,IGHD,IGHA1",
    celltype_key="cell_type_Scanorama",
    timepoints="TP1,TP2,TP3,TP4",
    col_wrap=2,
    out_name="ig_genes_violin_by_celltype_timepoint.png",
)

In [ ]:
genes = [g.strip() for g in args.genes.split(",") if g.strip()]
timepoints = [t.strip() for t in args.timepoints.split(",") if t.strip()]

print("=== Load data ===")
adata = sc.read_h5ad(args.drvi_input)
adata.X = adata.layers["log1p_norm"]

missing = [g for g in genes if g not in adata.var_names]
if missing:
    print(f"WARNING: not found in the dataset, skipped: {missing}")
genes = [g for g in genes if g in adata.var_names]
if not genes:
    raise RuntimeError("None of the specified genes were found in the dataset.")

### Derive timepoint from sample_id

In [ ]:
sample_id = adata.obs["sample_id"].astype(str)
split = sample_id.str.split(".", n=1, expand=True)
timepoint = np.where(split[1].notna(), "TP" + split[1], "control")
adata.obs["timepoint"] = timepoint

keep = pd.Series(timepoint, index=adata.obs_names).isin(timepoints).values
print(f"Filtering to timepoints {timepoints}: {keep.sum()} / {adata.n_obs} cells")
adata = adata[keep].copy()

### Long-format DataFrame for seaborn

In [ ]:
expr = adata[:, genes].X
expr = expr.toarray() if hasattr(expr, "toarray") else np.asarray(expr)
expr_df = pd.DataFrame(expr, columns=genes, index=adata.obs_names)
expr_df["cell_type"] = adata.obs[args.celltype_key].values
expr_df["timepoint"] = pd.Categorical(adata.obs["timepoint"].values, categories=timepoints, ordered=True)

long_df = expr_df.melt(id_vars=["cell_type", "timepoint"], value_vars=genes,
                        var_name="gene", value_name="expression")
long_df["gene"] = pd.Categorical(long_df["gene"], categories=genes, ordered=True)

### Plot

In [ ]:
sns.set_theme(style="whitegrid")
g = sns.catplot(
    data=long_df, kind="violin",
    x="cell_type", y="expression", hue="timepoint",
    col="gene", col_wrap=args.col_wrap,
    height=4, aspect=1.6, sharey=False,
    cut=0, inner=None, linewidth=0.5, density_norm="width",
)
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=90)
    ax.set_xlabel("")
g.set_titles("{col_name}")
g.set_ylabels("log1p-normalized expression")

out = os.path.join(args.output_dir, args.out_name)
g.savefig(out, bbox_inches="tight", dpi=130)
plt.close("all")
print(f"\nSaved: {out}")